In [1]:
import joblib
import pandas as pd
import sys
from pathlib import Path

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

from exoplanet_detector.config import DEFAULT_RUN_TAG, get_run_artifact_dirs

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
RUN_TAG = DEFAULT_RUN_TAG
run_dirs = get_run_artifact_dirs(RUN_TAG, create=True)

DEPLOYMENT_ARTIFACT_DIR = run_dirs["deployment"]
DEPLOY_MODELS_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_models.joblib"
DEPLOY_MANIFEST_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_manifest.csv"

deploy_bundle = joblib.load(DEPLOY_MODELS_PATH)
deploy_manifest = pd.read_csv(DEPLOY_MANIFEST_PATH)

deploy_manifest

,deploy_id,model,profile,threshold,koi_test_f2,koi_test_recall,koi_test_precision,k2p_f2,k2p_recall,k2p_precision
0,deploy_f2,knn,f2,0.363636,0.916696,0.958106,0.781575,0.885714,0.908621,0.804580
1,deploy_recall,rf,recall_constrained,0.029039,0.848823,0.998179,0.531008,0.921681,0.998276,0.705238
2,deploy_precision,hgb,precision_constrained,0.885432,0.565848,0.511840,0.979094,0.288149,0.244828,0.986111


In [3]:
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
X_koi_test = KOI_test_set.drop(columns=["label", "group_id"]).copy()
y_koi_test = KOI_test_set["label"].copy()
groups_koi_test = KOI_test_set["group_id"].copy()

X_k2p = K2P_set.drop(columns=["label", "group_id"]).copy()
y_k2p = K2P_set["label"].copy()
groups_k2p = K2P_set["group_id"].copy()

X_combined = pd.concat([X_koi_test, X_k2p], axis=0, ignore_index=True)
y_combined = pd.concat([y_koi_test, y_k2p], axis=0, ignore_index=True)
groups_combined = pd.concat([groups_koi_test, groups_k2p], axis=0, ignore_index=True)

evaluation_sets = {
    "KOI_test": (X_koi_test, y_koi_test),
    "K2P": (X_k2p, y_k2p),
    "KOI_test_plus_K2P": (X_combined, y_combined),
}